In [2]:
import numpy as np
import pandas as pd
import joblib
import time

import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    classification_report
)

In [3]:
X_train = joblib.load("../data/X_train_zero_day.pkl")
y_train = joblib.load("../data/y_train_zero_day.pkl")

X_test_seen = joblib.load("../data/X_test_seen.pkl")
y_test_seen = joblib.load("../data/y_test_seen.pkl")

X_test_zero_day = joblib.load("../data/X_test_zero_day.pkl")
y_test_zero_day = joblib.load("../data/y_test_zero_day.pkl")

print("Train:", X_train.shape)
print("Seen Test:", X_test_seen.shape)
print("Zero-Day Test:", X_test_zero_day.shape)

Train: (125973, 122)
Seen Test: (18794, 122)
Zero-Day Test: (3750, 122)


In [4]:
X_train_model, X_val, y_train_model, y_val = train_test_split(
    X_train,
    y_train,
    test_size=0.20,
    stratify=y_train,
    random_state=42
)

print("Training:", X_train_model.shape)
print("Validation:", X_val.shape)

print("\nTraining class distribution:")
print(pd.Series(y_train_model).value_counts())

print("\nValidation class distribution:")
print(pd.Series(y_val).value_counts())

Training: (100778, 122)
Validation: (25195, 122)

Training class distribution:
binary_label
0    53874
1    46904
Name: count, dtype: int64

Validation class distribution:
binary_label
0    13469
1    11726
Name: count, dtype: int64


In [5]:
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import LinearSVC
from sklearn.linear_model import LogisticRegression

In [6]:
models = {

    "Decision Tree": DecisionTreeClassifier(
        criterion="entropy",
        max_depth=40,
        min_samples_split=20,
        min_samples_leaf=1,
        max_features=None,
        random_state=42
    ),

    "Random Forest": RandomForestClassifier(
        n_estimators=250,
        max_depth=40,
        min_samples_split=2,
        min_samples_leaf=1,
        max_features=None,
        random_state=42,
        n_jobs=-1
    ),

    "KNN": KNeighborsClassifier(
        n_neighbors=3,
        weights="distance",
        metric="manhattan",
        n_jobs=-1
    ),

    "Linear SVM": LinearSVC(
        C=0.5,
        class_weight="balanced",
        max_iter=5000,
        random_state=42
    ),

    "Logistic Regression": LogisticRegression(
        C=10,
        class_weight=None,
        solver="liblinear",
        max_iter=1000,
        random_state=42
    )
}

In [7]:
trained_models = {}

for name, model in models.items():

    print(f"Training {name}...")

    start = time.time()

    model.fit(
        X_train_model,
        y_train_model
    )

    elapsed = time.time() - start

    trained_models[name] = model

    print(
        f"{name} trained in "
        f"{elapsed:.2f} seconds"
    )

Training Decision Tree...
Decision Tree trained in 1.95 seconds
Training Random Forest...
Random Forest trained in 199.34 seconds
Training KNN...
KNN trained in 0.03 seconds
Training Linear SVM...
Linear SVM trained in 6.40 seconds
Training Logistic Regression...
Logistic Regression trained in 13.57 seconds


In [8]:
def get_scores(model, X):

    if hasattr(model, "predict_proba"):
        return model.predict_proba(X)[:, 1]

    elif hasattr(model, "decision_function"):
        return model.decision_function(X)

    else:
        raise ValueError(
            "Model does not support probability or decision scores."
        )

In [ ]:
validation_scores = {}

for name, model in trained_models.items():

    print(f"Getting validation scores for {name}...")

    scores = get_scores(
        model,
        X_val
    )

    validation_scores[name] = scores

print("Validation scores generated.")

In [ ]:
thresholds = np.arange(
    0.10,
    0.91,
    0.01
)

threshold_results = []

for model_name, scores in validation_scores.items():

    for threshold in thresholds:

        y_pred = (
            scores >= threshold
        ).astype(int)

        precision = precision_score(
            y_val,
            y_pred,
            zero_division=0
        )

        recall = recall_score(
            y_val,
            y_pred,
            zero_division=0
        )

        f1 = f1_score(
            y_val,
            y_pred,
            zero_division=0
        )

        accuracy = accuracy_score(
            y_val,
            y_pred
        )

        cm = confusion_matrix(
            y_val,
            y_pred
        )

        tn, fp, fn, tp = cm.ravel()

        fpr = fp / (fp + tn)

        threshold_results.append({

            "Model": model_name,
            "Threshold": threshold,

            "Accuracy": accuracy,
            "Precision": precision,
            "Recall": recall,
            "F1": f1,

            "FPR": fpr
        })

threshold_results = pd.DataFrame(
    threshold_results
)

threshold_results.head()

In [ ]:
best_thresholds = (
    threshold_results
    .sort_values(
        ["Model", "F1"],
        ascending=[True, False]
    )
    .groupby("Model")
    .first()
    .reset_index()
)

best_thresholds

تمرکز روی RF و DT

In [ ]:
rf = RandomForestClassifier(
    n_estimators=250,
    max_depth=40,
    min_samples_split=2,
    min_samples_leaf=1,
    max_features=None,
    random_state=42,
    n_jobs=-1
)

rf.fit(
    X_train_model,
    y_train_model
)

print("Random Forest trained successfully.")

In [ ]:
val_scores = rf.predict_proba(X_val)[:, 1]

print("Validation scores generated.")
print("Min:", val_scores.min())
print("Max:", val_scores.max())
print("Mean:", val_scores.mean())

In [ ]:
thresholds = np.arange(
    0.05,
    0.96,
    0.01
)

results = []

for threshold in thresholds:

    y_pred = (
        val_scores >= threshold
    ).astype(int)

    tn, fp, fn, tp = confusion_matrix(
        y_val,
        y_pred
    ).ravel()

    accuracy = accuracy_score(
        y_val,
        y_pred
    )

    precision = precision_score(
        y_val,
        y_pred,
        zero_division=0
    )

    recall = recall_score(
        y_val,
        y_pred,
        zero_division=0
    )

    f1 = f1_score(
        y_val,
        y_pred,
        zero_division=0
    )

    fpr = fp / (fp + tn)

    results.append({
        "Threshold": threshold,
        "Accuracy": accuracy,
        "Precision": precision,
        "Recall": recall,
        "F1": f1,
        "FPR": fpr,
        "TP": tp,
        "FP": fp,
        "TN": tn,
        "FN": fn
    })

rf_threshold_results = pd.DataFrame(results)

rf_threshold_results.head()

In [ ]:
fpr_3 = rf_threshold_results[
    rf_threshold_results["FPR"] <= 0.03
].copy()

fpr_3.sort_values(
    "Recall",
    ascending=False
).head(10)

In [ ]:
fpr_5 = rf_threshold_results[
    rf_threshold_results["FPR"] <= 0.05
].copy()

fpr_5.sort_values(
    "Recall",
    ascending=False
).head(10)

In [ ]:
def best_threshold_under_fpr(
    results,
    max_fpr
):

    valid = results[
        results["FPR"] <= max_fpr
    ].copy()

    if len(valid) == 0:
        return None

    return (
        valid
        .sort_values(
            ["Recall", "F1"],
            ascending=False
        )
        .iloc[0]
    )

In [ ]:
best_3 = best_threshold_under_fpr(
    rf_threshold_results,
    0.03
)

best_5 = best_threshold_under_fpr(
    rf_threshold_results,
    0.05
)

best_10 = best_threshold_under_fpr(
    rf_threshold_results,
    0.10
)

print("FPR <= 3%")
print(best_3)

print("\nFPR <= 5%")
print(best_5)

print("\nFPR <= 10%")
print(best_10)

In [ ]:
print("Best threshold with FPR <= 3%:")
print(best_3)

print("\nBest threshold with FPR <= 5%:")
print(best_5)

print("\nBest threshold with FPR <= 10%:")
print(best_10)

In [ ]:
selected_threshold = best_5["Threshold"]

print(
    "Selected threshold:",
    selected_threshold
)

In [ ]:
seen_scores = rf.predict_proba(
    X_test_seen
)[:, 1]

y_pred_seen = (
    seen_scores >= selected_threshold
).astype(int)

tn, fp, fn, tp = confusion_matrix(
    y_test_seen,
    y_pred_seen
).ravel()

seen_accuracy = accuracy_score(
    y_test_seen,
    y_pred_seen
)

seen_precision = precision_score(
    y_test_seen,
    y_pred_seen,
    zero_division=0
)

seen_recall = recall_score(
    y_test_seen,
    y_pred_seen,
    zero_division=0
)

seen_f1 = f1_score(
    y_test_seen,
    y_pred_seen,
    zero_division=0
)

seen_fpr = fp / (fp + tn)

print("Seen Test Results")
print("-" * 40)

print(f"Accuracy : {seen_accuracy:.4f}")
print(f"Precision: {seen_precision:.4f}")
print(f"Recall   : {seen_recall:.4f}")
print(f"F1       : {seen_f1:.4f}")
print(f"FPR      : {seen_fpr:.4f}")

print("\nConfusion Matrix:")
print(confusion_matrix(
    y_test_seen,
    y_pred_seen
))

In [ ]:
zero_day_scores = rf.predict_proba(
    X_test_zero_day
)[:, 1]

y_pred_zero_day = (
    zero_day_scores >= selected_threshold
).astype(int)

zero_day_detection = (
    np.mean(
        y_pred_zero_day == 1
    ) * 100
)

print(
    f"Zero-Day Detection: "
    f"{zero_day_detection:.2f}%"
)

In [ ]:
baseline_rf = 44.11

improvement = (
    zero_day_detection
    - baseline_rf
)

print(
    f"Baseline RF: "
    f"{baseline_rf:.2f}%"
)

print(
    f"Optimized RF: "
    f"{zero_day_detection:.2f}%"
)

print(
    f"Improvement: "
    f"{improvement:+.2f} percentage points"
)

In [ ]:
candidate_thresholds = {
    "FPR <= 3%": best_3["Threshold"],
    "FPR <= 5%": best_5["Threshold"],
    "FPR <= 10%": best_10["Threshold"]
}

comparison = []

for scenario, threshold in candidate_thresholds.items():

    # Seen
    seen_pred = (
        seen_scores >= threshold
    ).astype(int)

    tn, fp, fn, tp = confusion_matrix(
        y_test_seen,
        seen_pred
    ).ravel()

    seen_fpr = fp / (fp + tn)

    seen_f1_value = f1_score(
        y_test_seen,
        seen_pred,
        zero_division=0
    )

    # Zero-Day
    zero_pred = (
        zero_day_scores >= threshold
    ).astype(int)

    zero_detection = (
        np.mean(zero_pred == 1)
        * 100
    )

    comparison.append({
        "Scenario": scenario,
        "Threshold": threshold,
        "Seen F1 (%)": seen_f1_value * 100,
        "Seen FPR (%)": seen_fpr * 100,
        "Zero-Day Detection (%)": zero_detection
    })

rf_threshold_comparison = pd.DataFrame(
    comparison
)

rf_threshold_comparison

In [ ]:
# ============================================================
# Decision Tree Optimization
# ============================================================

from sklearn.tree import DecisionTreeClassifier
from sklearn.model_selection import GridSearchCV

dt = DecisionTreeClassifier(
    random_state=42
)

dt_param_grid = {
    "criterion": ["gini", "entropy"],
    "max_depth": [10, 20, 30, 40, None],
    "min_samples_split": [2, 5, 10, 20],
    "min_samples_leaf": [1, 2, 5]
}

dt_grid = GridSearchCV(
    estimator=dt,
    param_grid=dt_param_grid,
    scoring="f1",
    cv=3,
    n_jobs=-1,
    verbose=1
)

start = time.time()

dt_grid.fit(
    X_train_model,
    y_train_model
)

elapsed = time.time() - start

print(f"Decision Tree tuning completed in {elapsed:.2f} seconds")

print("\nBest Parameters:")
print(dt_grid.best_params_)

print("\nBest Validation F1:")
print(dt_grid.best_score_)

In [ ]:
best_dt = dt_grid.best_estimator_

print("Best Decision Tree:")
print(best_dt)

In [ ]:
dt_val_scores = best_dt.predict_proba(
    X_val
)[:, 1]

print("Decision Tree validation scores generated.")

print("Min:", dt_val_scores.min())
print("Max:", dt_val_scores.max())
print("Mean:", dt_val_scores.mean())

In [ ]:
dt_thresholds = np.arange(
    0.05,
    0.96,
    0.01
)

dt_results = []

for threshold in dt_thresholds:

    y_pred = (
        dt_val_scores >= threshold
    ).astype(int)

    tn, fp, fn, tp = confusion_matrix(
        y_val,
        y_pred
    ).ravel()

    accuracy = accuracy_score(
        y_val,
        y_pred
    )

    precision = precision_score(
        y_val,
        y_pred,
        zero_division=0
    )

    recall = recall_score(
        y_val,
        y_pred,
        zero_division=0
    )

    f1 = f1_score(
        y_val,
        y_pred,
        zero_division=0
    )

    fpr = fp / (fp + tn)

    dt_results.append({
        "Threshold": threshold,
        "Accuracy": accuracy,
        "Precision": precision,
        "Recall": recall,
        "F1": f1,
        "FPR": fpr,
        "TP": tp,
        "FP": fp,
        "TN": tn,
        "FN": fn
    })

dt_threshold_results = pd.DataFrame(
    dt_results
)

dt_threshold_results.head()

In [ ]:
best_dt_f1 = (
    dt_threshold_results
    .sort_values(
        "F1",
        ascending=False
    )
    .iloc[0]
)

print("Best DT threshold based on F1:")
print(best_dt_f1)

In [ ]:
dt_best_3 = best_threshold_under_fpr(
    dt_threshold_results,
    0.03
)

dt_best_5 = best_threshold_under_fpr(
    dt_threshold_results,
    0.05
)

dt_best_10 = best_threshold_under_fpr(
    dt_threshold_results,
    0.10
)

print("DT - FPR <= 3%")
print(dt_best_3)

print("\nDT - FPR <= 5%")
print(dt_best_5)

print("\nDT - FPR <= 10%")
print(dt_best_10)

In [ ]:
dt_selected_threshold = dt_best_5["Threshold"]

print(
    "Selected DT threshold:",
    dt_selected_threshold
)

In [ ]:
dt_zero_day_scores = best_dt.predict_proba(
    X_test_zero_day
)[:, 1]

dt_zero_day_pred = (
    dt_zero_day_scores >= dt_selected_threshold
).astype(int)

dt_zero_day_detection = (
    np.mean(
        dt_zero_day_pred == 1
    ) * 100
)

print(
    f"Optimized DT Zero-Day Detection: "
    f"{dt_zero_day_detection:.2f}%"
)

In [ ]:
baseline_dt = 36.64

dt_improvement = (
    dt_zero_day_detection
    - baseline_dt
)

print(
    f"Baseline DT: "
    f"{baseline_dt:.2f}%"
)

print(
    f"Optimized DT: "
    f"{dt_zero_day_detection:.2f}%"
)

print(
    f"Improvement: "
    f"{dt_improvement:+.2f} percentage points"
)

In [ ]:
dt_seen_scores = best_dt.predict_proba(
    X_test_seen
)[:, 1]

dt_seen_pred = (
    dt_seen_scores >= dt_selected_threshold
).astype(int)

tn, fp, fn, tp = confusion_matrix(
    y_test_seen,
    dt_seen_pred
).ravel()

dt_seen_accuracy = accuracy_score(
    y_test_seen,
    dt_seen_pred
)

dt_seen_precision = precision_score(
    y_test_seen,
    dt_seen_pred,
    zero_division=0
)

dt_seen_recall = recall_score(
    y_test_seen,
    dt_seen_pred,
    zero_division=0
)

dt_seen_f1 = f1_score(
    y_test_seen,
    dt_seen_pred,
    zero_division=0
)

dt_seen_fpr = fp / (fp + tn)

print("Optimized Decision Tree - Seen Test")
print("-" * 45)

print(f"Accuracy : {dt_seen_accuracy:.4f}")
print(f"Precision: {dt_seen_precision:.4f}")
print(f"Recall   : {dt_seen_recall:.4f}")
print(f"F1       : {dt_seen_f1:.4f}")
print(f"FPR      : {dt_seen_fpr:.4f}")

print("\nConfusion Matrix:")
print(
    confusion_matrix(
        y_test_seen,
        dt_seen_pred
    )
)

In [ ]:
# ============================================================
# Regularized Decision Tree
# ============================================================

dt_regularized = DecisionTreeClassifier(
    criterion="entropy",
    max_depth=10,
    min_samples_split=20,
    min_samples_leaf=10,
    class_weight="balanced",
    random_state=42
)

start = time.time()

dt_regularized.fit(
    X_train_model,
    y_train_model
)

print(
    f"Regularized DT training completed in "
    f"{time.time() - start:.2f} seconds"
)

In [ ]:
dt_reg_val_scores = dt_regularized.predict_proba(
    X_val
)[:, 1]

In [ ]:
dt_reg_results = []

for threshold in np.arange(0.01, 0.96, 0.01):

    y_pred = (
        dt_reg_val_scores >= threshold
    ).astype(int)

    tn, fp, fn, tp = confusion_matrix(
        y_val,
        y_pred
    ).ravel()

    fpr = fp / (fp + tn)

    dt_reg_results.append({
        "Threshold": threshold,
        "Accuracy": accuracy_score(y_val, y_pred),
        "Precision": precision_score(
            y_val,
            y_pred,
            zero_division=0
        ),
        "Recall": recall_score(
            y_val,
            y_pred,
            zero_division=0
        ),
        "F1": f1_score(
            y_val,
            y_pred,
            zero_division=0
        ),
        "FPR": fpr
    })

dt_reg_results = pd.DataFrame(
    dt_reg_results
)

dt_reg_best = best_threshold_under_fpr(
    dt_reg_results,
    0.05
)

print(dt_reg_best)

In [ ]:
dt_reg_threshold = dt_reg_best["Threshold"]

dt_reg_zero_scores = dt_regularized.predict_proba(
    X_test_zero_day
)[:, 1]

dt_reg_zero_pred = (
    dt_reg_zero_scores >= dt_reg_threshold
).astype(int)

dt_reg_zero_detection = (
    np.mean(
        dt_reg_zero_pred == 1
    ) * 100
)

print(
    f"Regularized DT Zero-Day Detection: "
    f"{dt_reg_zero_detection:.2f}%"
)

In [ ]:
# ============================================================
# Regularized DT - Seen Test Evaluation
# ============================================================

dt_reg_seen_scores = dt_regularized.predict_proba(
    X_test_seen
)[:, 1]

dt_reg_seen_pred = (
    dt_reg_seen_scores >= dt_reg_threshold
).astype(int)

tn, fp, fn, tp = confusion_matrix(
    y_test_seen,
    dt_reg_seen_pred
).ravel()

dt_reg_seen_accuracy = accuracy_score(
    y_test_seen,
    dt_reg_seen_pred
)

dt_reg_seen_precision = precision_score(
    y_test_seen,
    dt_reg_seen_pred,
    zero_division=0
)

dt_reg_seen_recall = recall_score(
    y_test_seen,
    dt_reg_seen_pred,
    zero_division=0
)

dt_reg_seen_f1 = f1_score(
    y_test_seen,
    dt_reg_seen_pred,
    zero_division=0
)

dt_reg_seen_fpr = fp / (fp + tn)

print("Regularized Decision Tree - Seen Test")
print("-" * 50)

print(f"Accuracy : {dt_reg_seen_accuracy:.4f}")
print(f"Precision: {dt_reg_seen_precision:.4f}")
print(f"Recall   : {dt_reg_seen_recall:.4f}")
print(f"F1       : {dt_reg_seen_f1:.4f}")
print(f"FPR      : {dt_reg_seen_fpr:.4f}")

print("\nConfusion Matrix:")
print(
    confusion_matrix(
        y_test_seen,
        dt_reg_seen_pred
    )
)

In [ ]:
print("\n" + "=" * 50)
print("DECISION TREE COMPARISON")
print("=" * 50)

print(f"Baseline DT Zero-Day      : {baseline_dt:.2f}%")
print(
    f"Original Optimized DT     : "
    f"{dt_zero_day_detection:.2f}%"
)
print(
    f"Regularized DT Zero-Day   : "
    f"{dt_reg_zero_detection:.2f}%"
)

print(
    f"\nRegularized Improvement   : "
    f"{dt_reg_zero_detection - baseline_dt:+.2f} pp"
)

In [ ]:
# ============================================
# Logistic Regression - Baseline
# ============================================

from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix
)

import numpy as np
import pandas as pd
import time


# Create baseline Logistic Regression model
lr_baseline = LogisticRegression(
    max_iter=2000,
    random_state=42
)

# Train
start_time = time.time()

lr_baseline.fit(X_train_model, y_train_model)

lr_baseline_time = time.time() - start_time

print(f"Training Time: {lr_baseline_time:.2f} seconds")

In [ ]:
# ============================================
# Logistic Regression - Baseline Seen Test
# ============================================

y_pred_lr_baseline_seen = lr_baseline.predict(X_test_seen)

lr_baseline_seen_accuracy = accuracy_score(
    y_test_seen,
    y_pred_lr_baseline_seen
)

lr_baseline_seen_precision = precision_score(
    y_test_seen,
    y_pred_lr_baseline_seen,
    zero_division=0
)

lr_baseline_seen_recall = recall_score(
    y_test_seen,
    y_pred_lr_baseline_seen,
    zero_division=0
)

lr_baseline_seen_f1 = f1_score(
    y_test_seen,
    y_pred_lr_baseline_seen,
    zero_division=0
)

print("========== Logistic Regression Baseline - Seen Test ==========")
print(f"Accuracy : {lr_baseline_seen_accuracy:.4f}")
print(f"Precision: {lr_baseline_seen_precision:.4f}")
print(f"Recall   : {lr_baseline_seen_recall:.4f}")
print(f"F1 Score : {lr_baseline_seen_f1:.4f}")

In [ ]:
# ============================================
# Final Visualization 1
# Zero-Day Detection Rate Comparison
# ============================================

final_zero_day_results = pd.DataFrame({
    "Model": [
        "Random Forest",
        "Decision Tree",
        "KNN"
    ],
    "Zero-Day Detection (%)": [
        rf_zero_day_detection,
        dt_zero_day_detection,
        40.77
    ]
})

plt.figure(figsize=(10, 6))

bars = plt.bar(
    final_zero_day_results["Model"],
    final_zero_day_results["Zero-Day Detection (%)"]
)

plt.ylabel("Zero-Day Detection Rate (%)")
plt.xlabel("Model")
plt.title("Zero-Day Detection Rate of Final Models")
plt.ylim(0, 100)

for bar in bars:
    height = bar.get_height()
    plt.text(
        bar.get_x() + bar.get_width() / 2,
        height + 1,
        f"{height:.2f}%",
        ha="center",
        va="bottom"
    )

plt.grid(axis="y", alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
# ============================================
# Final Visualization 2
# Before vs After Optimization
# ============================================

optimization_comparison = pd.DataFrame({
    "Model": [
        "Random Forest",
        "Decision Tree"
    ],
    "Baseline": [
        44.11,
        36.64
    ],
    "Optimized": [
        56.91,
        50.03
    ]
})

x = np.arange(len(optimization_comparison))
width = 0.35

plt.figure(figsize=(10, 6))

bars1 = plt.bar(
    x - width / 2,
    optimization_comparison["Baseline"],
    width,
    label="Baseline"
)

bars2 = plt.bar(
    x + width / 2,
    optimization_comparison["Optimized"],
    width,
    label="Optimized"
)

plt.ylabel("Zero-Day Detection Rate (%)")
plt.xlabel("Model")
plt.title("Effect of Optimization on Zero-Day Detection")
plt.xticks(x, optimization_comparison["Model"])
plt.ylim(0, 100)
plt.legend()

for bars in [bars1, bars2]:
    for bar in bars:
        height = bar.get_height()
        plt.text(
            bar.get_x() + bar.get_width() / 2,
            height + 1,
            f"{height:.2f}%",
            ha="center",
            va="bottom"
        )

plt.grid(axis="y", alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
# ============================================
# Final Visualization 3
# Seen Test Performance Comparison
# ============================================

seen_test_comparison = pd.DataFrame({
    "Metric": [
        "Accuracy",
        "Precision",
        "Recall",
        "F1-Score",
        "FPR"
    ],
    "Random Forest": [
        rf_seen_results["Accuracy"],
        rf_seen_results["Precision"],
        rf_seen_results["Recall"],
        rf_seen_results["F1"],
        rf_seen_results["FPR"]
    ],
    "Decision Tree": [
        dt_seen_results["Accuracy"],
        dt_seen_results["Precision"],
        dt_seen_results["Recall"],
        dt_seen_results["F1"],
        dt_seen_results["FPR"]
    ]
})

x = np.arange(len(seen_test_comparison))
width = 0.35

plt.figure(figsize=(11, 6))

bars1 = plt.bar(
    x - width / 2,
    seen_test_comparison["Random Forest"] * 100,
    width,
    label="Random Forest"
)

bars2 = plt.bar(
    x + width / 2,
    seen_test_comparison["Decision Tree"] * 100,
    width,
    label="Decision Tree"
)

plt.ylabel("Percentage (%)")
plt.xlabel("Metric")
plt.title("Seen Test Performance of Final Models")
plt.xticks(x, seen_test_comparison["Metric"])
plt.ylim(0, 105)
plt.legend()

for bars in [bars1, bars2]:
    for bar in bars:
        height = bar.get_height()
        plt.text(
            bar.get_x() + bar.get_width() / 2,
            height + 1,
            f"{height:.2f}%",
            ha="center",
            va="bottom"
        )

plt.grid(axis="y", alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
# ============================================
# Final Visualization 4
# Random Forest - Threshold Analysis
# ============================================

rf_threshold_plot = rf_threshold_results.copy()

plt.figure(figsize=(10, 6))

plt.plot(
    rf_threshold_plot["Threshold"],
    rf_threshold_plot["FPR"] * 100,
    marker="o",
    label="FPR"
)

plt.plot(
    rf_threshold_plot["Threshold"],
    rf_threshold_plot["Zero-Day Detection"] ,
    marker="s",
    label="Zero-Day Detection"
)

plt.xlabel("Threshold")
plt.ylabel("Percentage (%)")
plt.title("Random Forest: Threshold vs FPR and Zero-Day Detection")
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
baseline_zero_day = pd.DataFrame({

    "Model": [
        "Decision Tree",
        "Random Forest",
        "KNN",
        "Linear SVM",
        "Logistic Regression"
    ],

    "Default Threshold Zero-Day (%)": [
        36.64,
        44.11,
        40.77,
        37.01,
        40.91
    ]
})

In [ ]:
threshold_comparison = baseline_zero_day.merge(
    zero_day_threshold_results,
    on="Model"
)

threshold_comparison[
    "Improvement (pp)"
] = (
    threshold_comparison[
        "Zero-Day Detection (%)"
    ]
    -
    threshold_comparison[
        "Default Threshold Zero-Day (%)"
    ]
)

threshold_comparison